In [ ]:
import json
from bs4 import BeautifulSoup
from shapely.geometry import Polygon

In [ ]:
geodata = json.load(open("/home/dkhoa/code/LightningCat/dataAndDataCleanup/SportSGSportFacilitiesGEOJSON.geojson"))

In [ ]:
list(geodata.keys())

In [ ]:
d = geodata['features'][0]['properties']['Description']

In [ ]:
raw_data = geodata['features']

In [ ]:
len(geodata['features'])

In [ ]:
parsed = BeautifulSoup(d, 'html.parser')

In [ ]:
attributes = parsed.center.table.find_all('tr')[1:]

In [ ]:
attr_names = {tr.th.text: tr.td.text for tr in attributes}

In [ ]:
attr_names

In [ ]:
def get_list_facility_types(facilities):
    facility_types = set()
    for facility in facilities:
        html_content = facility['properties']['Description']
        parsed = BeautifulSoup(d, 'html.parser')
        raw_attributes = parsed.center.table.find_all('tr')[1:]
        attributes = {tr.th.text: tr.td.text for tr in raw_attributes}
        facility_types.add(attributes['FACILITIES'])
    return facility_types

In [ ]:
facility_types = get_list_facility_types(facilities)

In [ ]:
facility_types

In [ ]:
raw_description = [r['properties']['Description'] for r in raw_data]
raw_coordinates = [r['geometry']['coordinates'] for r in raw_data]

In [ ]:
raw_description[0]

In [ ]:
raw_data[0]

List of thing to collect:
- Facility Name.
- Activities.
- Age Range
- Lat/Long
- Type
- Address
- Geojson
- Operation Time
- Contact

In [ ]:
activity_map = {
"SWIMMING_C":	"Swimming",
"FOOTBALL_S":	"Football",
"TENNIS_SQU":	"Tennis",
"GYM_OPERAT":	"Gym",
"WADING_POO":	"Wading",
"INDOOR_SPO":	"Indoor",
"BADMINTON_":	"Badminton",
"TABLE_TENN":	"Table_tennis",
"NETBALL_CO":	"Netball",
"VOLLEYBALL":	"Volleyball",
"BASKETBALL":	"Basketball",
"ATHLETICS_":	"Athletics",
"FOOTBALL_F":	"Football",
"SOCCER_COU":	"Soccer",
"RUGBY_FIEL":	"Rugby",
"TENNIS_COU":	"Tennis",
"SQUASH_COU":	"Squash",
"GYM"       :   "Gym",
"HOCKEY_PIT":	"Hockey",
"PETANQUE_C":	"Petanque",
"GATEBALL_C":	"Gateball",
"LAWN_BOWL_":	"Lawn_bowl",
"PICKLEBALL":	"Pickleball",
}
def get_activities(ats):
    return list(set([activity_map[k] for k, v in ats.items() if k in activity_map and v != '']))

In [ ]:
json_data = []
for raw, raw_coords in zip(raw_description, raw_coordinates):
    p = BeautifulSoup(raw, 'html.parser')
    d = {}
    raw_attributes = p.center.table.find_all('tr')[1:]
    a = {tr.th.text: tr.td.text for tr in raw_attributes}
    d["name"] = a["SPORTS_CEN"]
    d["address"] = f"{a['HOUSE_BLOC']} {a['ROAD_NAME']}, {a['POSTAL_COD']}"
    d["contact"] = a["CONTACT_NO"]
    d["time"] = a["STADIUM_OP"]
    d["age_range"] = ["Children", "Adult", "Senior"]
    d["activities"] = get_activities(a)
    d["coordinates"] = [[c[0], c[1]] for c in raw_coords[0]]
    poly = Polygon(d["coordinates"])
    d["location"] = [poly.centroid.x, poly.centroid.y]
    if d["name"] == "Burghley Squash and Tennis Centre":
        print(a)
    json_data.append(d)

In [ ]:
json.dump(json_data, open("facility_data.json", "w"), indent=4)